In [ ]:
# pip install pg8000 urllib3 python-dotenv
# pip install -U certifi urllib3

### 전체 컬럼 backfill

In [5]:
import os
import json
import time
import random
from datetime import datetime, timedelta, date
from urllib.parse import quote

import pg8000.native
import urllib3

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass


# ============================================================
# CONFIG
# ============================================================
FB_API_VERSION = os.getenv("FB_API_VERSION", "v18.0")
INSIGHTS_BATCH_SIZE = int(os.getenv("INSIGHTS_BATCH_SIZE", "50"))
MAX_RETRIES = int(os.getenv("MAX_RETRIES", "5"))
BASE_BACKOFF = float(os.getenv("BASE_BACKOFF", "1.0"))
HTTP_TIMEOUT_CONNECT = float(os.getenv("HTTP_TIMEOUT_CONNECT", "30"))
HTTP_TIMEOUT_READ = float(os.getenv("HTTP_TIMEOUT_READ", "120"))
SLEEP_BETWEEN_CALLS_MIN = float(os.getenv("SLEEP_BETWEEN_CALLS_MIN", "0.05"))
SLEEP_BETWEEN_CALLS_MAX = float(os.getenv("SLEEP_BETWEEN_CALLS_MAX", "0.25"))

# 한 번의 API 호출로 기본 성과 + 전환 + 참여 + 영상 지표 전부 수집
ALL_FIELDS = (
    "ad_id,date_start,"
    # 기본 성과
    "reach,impressions,clicks,ctr,inline_post_engagement,"
    # 비용·효율
    "spend,frequency,cpc,cpm,purchase_roas,"
    # 전환·참여 (actions 배열에서 파싱)
    "actions,"
    # 영상 시청 구간 (별도 필드)
    "video_p25_watched_actions,video_p50_watched_actions,"
    "video_p75_watched_actions,video_p100_watched_actions,"
    "video_thruplay_watched_actions"
)

ACTION_KEYS = {
    "link_clicks"               : ["link_click", "inline_link_click", "outbound_click", "outbound_clicks"],
    "website_landing_page_views": ["landing_page_view", "omni_landing_page_view"],
    "add_to_cart"               : ["add_to_cart", "offsite_conversion.fb_pixel_add_to_cart", "omni_add_to_cart"],
    "purchases"                 : ["purchase", "offsite_conversion.fb_pixel_purchase", "omni_purchase"],
    "view_content"              : ["view_content", "offsite_conversion.fb_pixel_view_content", "omni_view_content"],
    "initiate_checkout"         : ["initiate_checkout", "offsite_conversion.fb_pixel_initiate_checkout", "omni_initiated_checkout"],
    "complete_registration"     : ["complete_registration", "offsite_conversion.fb_pixel_complete_registration"],
    "instagram_profile_visits"  : ["instagram_profile_visit"],
    "follows"                   : ["follow", "page_like"],
    "post_reactions"            : ["post_reaction"],
    "comments"                  : ["comment"],
    "post_saves"                : ["onsite_conversion.post_save"],
    "video_views"               : ["video_view"],
}


# ============================================================
# UTIL
# ============================================================
def chunk_list(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

def daterange(start: date, end_inclusive: date):
    d = start
    while d <= end_inclusive:
        yield d
        d += timedelta(days=1)

def normalize_act_id(s: str) -> str:
    s = (s or "").strip()
    return s if s.startswith("act_") else f"act_{s}"

def is_permission_error(err: dict) -> bool:
    return bool(err) and err.get("code") == 200

def is_rate_limit_error(err: dict) -> bool:
    if not err: return False
    code = err.get("code")
    msg = (err.get("message") or "").lower()
    return code in (4, 17, 32, 613) or "rate limit" in msg or "too many calls" in msg

def is_transient_http_status(status: int) -> bool:
    return status in (408, 429, 500, 502, 503, 504)

def safe_int(v):
    try: return int(float(v))
    except: return 0

def safe_float(v):
    try: return float(v)
    except: return 0.0

def parse_actions(actions_list):
    out = {}
    for a in (actions_list or []):
        k, v = a.get("action_type"), a.get("value")
        if k and v is not None:
            try: out[k] = float(v)
            except: pass
    return out

def max_action(actions_map, candidates):
    vals = [actions_map[k] for k in candidates if k in actions_map]
    return int(max(vals)) if vals else 0

def parse_video_pct_field(field_data):
    """video_p25_watched_actions 등 → int"""
    for item in (field_data or []):
        if isinstance(item, dict) and item.get("action_type") == "video_view":
            try: return int(float(item.get("value", 0)))
            except: pass
    return 0

def parse_purchase_roas(v):
    if v is None: return 0.0
    if isinstance(v, list):
        return safe_float(v[0]["value"]) if v else 0.0
    return safe_float(v)


# ============================================================
# DB
# ============================================================
def get_db_conn():
    return pg8000.native.Connection(
        user=os.environ["DB_USER"], password=os.environ["DB_PASSWORD"],
        host=os.environ["DB_HOST"], database=os.environ["DB_NAME"], port=5432
    )

def get_ad_accounts(conn):
    rows = conn.run("SELECT fb_ad_account_id FROM ad_account WHERE fb_ad_account_id IS NOT NULL")
    return [str(r[0]).strip() for r in rows if r and r[0] is not None]

def get_ads_by_account(conn, fb_ad_account_id):
    rows = conn.run("""
        SELECT ad_id, fb_ad_id FROM ad
        WHERE account_id = (SELECT account_id FROM ad_account WHERE fb_ad_account_id = :aid)
        AND fb_ad_id IS NOT NULL
    """, aid=fb_ad_account_id)
    return [{"ad_id": r[0], "fb_ad_id": str(r[1]).strip()} for r in rows]

def upsert_all(conn, rows, ad_id_map, created_at_utc):
    """모든 컬럼 upsert (기본 성과 + 전환 + 참여 + 영상)"""
    saved = skipped = 0
    for rec in rows:
        fb_ad_id = rec.get("fb_ad_id")
        if not fb_ad_id or fb_ad_id not in ad_id_map:
            skipped += 1; continue
        try:
            conn.run("""
                INSERT INTO ad_performance_daily (
                    ad_id, date, age, gender,
                    reach, impressions, clicks, ctr, inline_post_engagement,
                    spend, frequency, cpc, cpm, purchase_roas,
                    link_clicks, website_landing_page_views, add_to_cart, purchases,
                    view_content, initiate_checkout, complete_registration,
                    instagram_profile_visits, follows, post_reactions, comments, post_saves,
                    video_views, video_p25_watched, video_p50_watched, video_p75_watched,
                    video_p100_watched, video_thruplay_watched,
                    created_at, updated_at
                ) VALUES (
                    :ad_id, :date, :age, :gender,
                    :reach, :impressions, :clicks, :ctr, :inline_post_engagement,
                    :spend, :frequency, :cpc, :cpm, :purchase_roas,
                    :link_clicks, :website_landing_page_views, :add_to_cart, :purchases,
                    :view_content, :initiate_checkout, :complete_registration,
                    :instagram_profile_visits, :follows, :post_reactions, :comments, :post_saves,
                    :video_views, :video_p25_watched, :video_p50_watched, :video_p75_watched,
                    :video_p100_watched, :video_thruplay_watched,
                    :created_at, :updated_at
                )
                ON CONFLICT (ad_id, date, age, gender) DO UPDATE SET
                    reach                        = EXCLUDED.reach,
                    impressions                  = EXCLUDED.impressions,
                    clicks                       = EXCLUDED.clicks,
                    ctr                          = EXCLUDED.ctr,
                    inline_post_engagement       = EXCLUDED.inline_post_engagement,
                    spend                        = EXCLUDED.spend,
                    frequency                    = EXCLUDED.frequency,
                    cpc                          = EXCLUDED.cpc,
                    cpm                          = EXCLUDED.cpm,
                    purchase_roas                = EXCLUDED.purchase_roas,
                    link_clicks                  = EXCLUDED.link_clicks,
                    website_landing_page_views   = EXCLUDED.website_landing_page_views,
                    add_to_cart                  = EXCLUDED.add_to_cart,
                    purchases                    = EXCLUDED.purchases,
                    view_content                 = EXCLUDED.view_content,
                    initiate_checkout            = EXCLUDED.initiate_checkout,
                    complete_registration        = EXCLUDED.complete_registration,
                    instagram_profile_visits     = EXCLUDED.instagram_profile_visits,
                    follows                      = EXCLUDED.follows,
                    post_reactions               = EXCLUDED.post_reactions,
                    comments                     = EXCLUDED.comments,
                    post_saves                   = EXCLUDED.post_saves,
                    video_views                  = EXCLUDED.video_views,
                    video_p25_watched            = EXCLUDED.video_p25_watched,
                    video_p50_watched            = EXCLUDED.video_p50_watched,
                    video_p75_watched            = EXCLUDED.video_p75_watched,
                    video_p100_watched           = EXCLUDED.video_p100_watched,
                    video_thruplay_watched       = EXCLUDED.video_thruplay_watched,
                    updated_at                   = :updated_at
            """,
            ad_id=ad_id_map[fb_ad_id], date=rec["date_start"],
            age=rec["age"], gender=rec["gender"],
            reach=rec["reach"], impressions=rec["impressions"],
            clicks=rec["clicks"], ctr=rec["ctr"],
            inline_post_engagement=rec["inline_post_engagement"],
            spend=rec["spend"], frequency=rec["frequency"],
            cpc=rec["cpc"], cpm=rec["cpm"], purchase_roas=rec["purchase_roas"],
            link_clicks=rec["link_clicks"],
            website_landing_page_views=rec["website_landing_page_views"],
            add_to_cart=rec["add_to_cart"], purchases=rec["purchases"],
            view_content=rec["view_content"], initiate_checkout=rec["initiate_checkout"],
            complete_registration=rec["complete_registration"],
            instagram_profile_visits=rec["instagram_profile_visits"],
            follows=rec["follows"], post_reactions=rec["post_reactions"],
            comments=rec["comments"], post_saves=rec["post_saves"],
            video_views=rec["video_views"],
            video_p25_watched=rec["video_p25_watched"],
            video_p50_watched=rec["video_p50_watched"],
            video_p75_watched=rec["video_p75_watched"],
            video_p100_watched=rec["video_p100_watched"],
            video_thruplay_watched=rec["video_thruplay_watched"],
            created_at=created_at_utc, updated_at=created_at_utc)
            saved += 1
        except Exception as e:
            skipped += 1
            print(f"⚠️ upsert failed fb_ad_id={fb_ad_id} date={rec.get('date_start')}: {e}")
    return saved, skipped


# ============================================================
# META API
# ============================================================
def fetch_batch(http, access_token, act_id, fb_ad_ids, since, until):
    base_url = f"https://graph.facebook.com/{FB_API_VERSION}/{act_id}/insights"
    params = {
        "access_token"  : access_token,
        "level"         : "ad",
        "breakdowns"    : "age,gender",
        "time_range"    : json.dumps({"since": since, "until": until}),
        "time_increment": "1",
        "fields"        : ALL_FIELDS,
        "filtering"     : json.dumps([{"field": "ad.id", "operator": "IN", "value": fb_ad_ids}]),
        "limit"         : "5000",
    }

    def build_url(extra=None):
        p = {**params, **(extra or {})}
        return base_url + "?" + "&".join(f"{k}={quote(str(v))}" for k, v in p.items())

    backoff = BASE_BACKOFF
    for attempt in range(MAX_RETRIES):
        try:
            resp = http.request("GET", build_url())
            data = json.loads(resp.data.decode("utf-8", errors="replace"))
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                s = backoff + random.uniform(0, 0.5)
                print(f"⚠️ Network error (attempt {attempt+1}): {e} → sleep {s:.1f}s")
                time.sleep(s); backoff *= 2; continue
            return [], "FAILED"

        if resp.status == 200 and "data" in data:
            rows = _normalize(data["data"], since)
            while True:
                after = ((data.get("paging") or {}).get("cursors") or {}).get("after")
                if not after: break
                resp2 = http.request("GET", build_url({"after": after}))
                data2 = json.loads(resp2.data.decode("utf-8", errors="replace"))
                if resp2.status != 200 or "data" not in data2: break
                rows.extend(_normalize(data2["data"], since))
                data = data2
            return rows, "OK"

        err = data.get("error") or {}
        if resp.status == 403 and is_permission_error(err):
            return [], "PERMISSION"
        if is_rate_limit_error(err) or is_transient_http_status(resp.status):
            if attempt < MAX_RETRIES - 1:
                s = backoff + random.uniform(0, 0.5)
                time.sleep(s); backoff *= 2; continue
        print(f"⚠️ Non-retryable error status={resp.status}: {str(data)[:200]}")
        return [], "FAILED"
    return [], "FAILED"


def _normalize(data_rows, forced_date):
    out = []
    for ins in data_rows or []:
        fb_ad_id = str(ins.get("ad_id") or "").strip()
        if not fb_ad_id:
            continue
        am = parse_actions(ins.get("actions"))
        out.append({
            "fb_ad_id"  : fb_ad_id,
            # API date_start 무시 — 계정 타임존(미국) 기준 반환으로 KST 기준 -1일 오류 발생
            # Lambda와 동일하게 요청 날짜(since)를 강제 사용
            "date_start": forced_date,
            "age"       : ins.get("age") or "unknown",
            "gender"    : ins.get("gender") or "unknown",
            # 기본 성과
            "reach"                 : safe_int(ins.get("reach")),
            "impressions"           : safe_int(ins.get("impressions")),
            "clicks"                : safe_int(ins.get("clicks")),
            "ctr"                   : safe_float(ins.get("ctr")),
            "inline_post_engagement": safe_int(ins.get("inline_post_engagement")),
            # 비용·효율
            "spend"         : safe_float(ins.get("spend")),
            "frequency"     : safe_float(ins.get("frequency")),
            "cpc"           : safe_float(ins.get("cpc")),
            "cpm"           : safe_float(ins.get("cpm")),
            "purchase_roas" : parse_purchase_roas(ins.get("purchase_roas")),
            # 전환 지표
            "link_clicks"                : max_action(am, ACTION_KEYS["link_clicks"]),
            "website_landing_page_views" : max_action(am, ACTION_KEYS["website_landing_page_views"]),
            "add_to_cart"                : max_action(am, ACTION_KEYS["add_to_cart"]),
            "purchases"                  : max_action(am, ACTION_KEYS["purchases"]),
            "view_content"               : max_action(am, ACTION_KEYS["view_content"]),
            "initiate_checkout"          : max_action(am, ACTION_KEYS["initiate_checkout"]),
            "complete_registration"      : max_action(am, ACTION_KEYS["complete_registration"]),
            # 참여 지표
            "instagram_profile_visits"   : max_action(am, ACTION_KEYS["instagram_profile_visits"]),
            "follows"                    : max_action(am, ACTION_KEYS["follows"]),
            "post_reactions"             : max_action(am, ACTION_KEYS["post_reactions"]),
            "comments"                   : max_action(am, ACTION_KEYS["comments"]),
            "post_saves"                 : max_action(am, ACTION_KEYS["post_saves"]),
            "video_views"                : max_action(am, ACTION_KEYS["video_views"]),
            # 영상 시청 구간
            "video_p25_watched"     : parse_video_pct_field(ins.get("video_p25_watched_actions")),
            "video_p50_watched"     : parse_video_pct_field(ins.get("video_p50_watched_actions")),
            "video_p75_watched"     : parse_video_pct_field(ins.get("video_p75_watched_actions")),
            "video_p100_watched"    : parse_video_pct_field(ins.get("video_p100_watched_actions")),
            "video_thruplay_watched": parse_video_pct_field(ins.get("video_thruplay_watched_actions")),
        })
    return out


# ============================================================
# MAIN BACKFILL
# ============================================================
def backfill(start_date: str, end_date: str):
    """지정 기간의 전체 컬럼 backfill (기본 성과 + 전환 + 참여 + 영상)."""
    access_token = os.environ["META_ACCESS_TOKEN"]
    start = datetime.strptime(start_date, "%Y-%m-%d").date()
    end   = datetime.strptime(end_date,   "%Y-%m-%d").date()

    db   = get_db_conn()
    http = urllib3.PoolManager(
        timeout=urllib3.Timeout(connect=HTTP_TIMEOUT_CONNECT, read=HTTP_TIMEOUT_READ),
        retries=False
    )

    try:
        accounts = get_ad_accounts(db)
        print(f"✅ ad_accounts: {len(accounts)}")

        account_ads = []
        for fb_act_id in accounts:
            ads = get_ads_by_account(db, fb_act_id)
            ad_id_map = {a["fb_ad_id"]: a["ad_id"] for a in ads}
            account_ads.append((fb_act_id, normalize_act_id(fb_act_id), list(ad_id_map.keys()), ad_id_map))
        print("✅ Ads loaded.")

        total_saved = total_rows = perm_skipped = 0

        for d in daterange(start, end):
            since = d.strftime("%Y-%m-%d")
            until = (d + timedelta(days=1)).strftime("%Y-%m-%d")
            print(f"\n📅 {since}")
            day_saved = day_rows = 0
            now_utc = datetime.utcnow()

            for fb_act_id, act_id, fb_ad_ids, ad_id_map in account_ads:
                if not fb_ad_ids: continue
                perm_denied = False
                for batch in chunk_list(fb_ad_ids, INSIGHTS_BATCH_SIZE):
                    rows, reason = fetch_batch(http, access_token, act_id, batch, since, until)
                    if reason == "PERMISSION":
                        perm_denied = True; break
                    if rows:
                        saved, _ = upsert_all(db, rows, ad_id_map, now_utc)
                        day_saved += saved; total_saved += saved
                        day_rows  += len(rows); total_rows += len(rows)
                    time.sleep(random.uniform(SLEEP_BETWEEN_CALLS_MIN, SLEEP_BETWEEN_CALLS_MAX))
                if perm_denied:
                    perm_skipped += 1
                    print(f"  🚫 Permission: {act_id}")

            print(f"  ✅ fetched={day_rows} saved={day_saved}")

        print(f"\n{'='*40}")
        print(f"✅ Backfill done.  total_fetched={total_rows}  total_saved={total_saved}  perm_skipped={perm_skipped}")

    finally:
        try: http.clear()
        except: pass
        try: db.close()
        except: pass


# 실행
backfill("2026-05-18", "2026-05-26")

✅ ad_accounts: 41
✅ Ads loaded.

📅 2026-05-18
  🚫 Permission: act_399481461969972
  🚫 Permission: act_4241077792781592
  🚫 Permission: act_708046804726689
  🚫 Permission: act_601392272220010
  🚫 Permission: act_911591441881884
  🚫 Permission: act_3747512468894152
  ✅ fetched=1276 saved=1276

📅 2026-05-19
  🚫 Permission: act_399481461969972
  🚫 Permission: act_4241077792781592
  🚫 Permission: act_708046804726689
  🚫 Permission: act_601392272220010
  🚫 Permission: act_911591441881884
  🚫 Permission: act_3747512468894152
  ✅ fetched=1144 saved=1144

📅 2026-05-20
  🚫 Permission: act_399481461969972
  🚫 Permission: act_4241077792781592
  🚫 Permission: act_708046804726689
  🚫 Permission: act_601392272220010
  🚫 Permission: act_911591441881884
  🚫 Permission: act_3747512468894152
  ✅ fetched=1043 saved=1043

📅 2026-05-21
  🚫 Permission: act_399481461969972
  🚫 Permission: act_4241077792781592
  🚫 Permission: act_708046804726689
  🚫 Permission: act_601392272220010
  🚫 Permission: act_911591441

### 성과 컬럼 backfill (reach, impressions, clicks, ctr, inline_post_engagement 제외)

In [6]:
import os
import json
import time
import random
from datetime import datetime, timedelta, date
from urllib.parse import quote

import pg8000.native
import urllib3

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

"""
전환 / 참여 / 영상 지표 backfill
──────────────────────────────────────────────────────────────
대상 컬럼 (기본 성과 지표 제외):
  spend, frequency, cpc, cpm, purchase_roas
  link_clicks, website_landing_page_views, add_to_cart, purchases
  view_content, initiate_checkout, complete_registration
  instagram_profile_visits, follows, post_reactions, comments, post_saves
  video_views, video_p25_watched, video_p50_watched, video_p75_watched,
  video_p100_watched, video_thruplay_watched
"""

FB_API_VERSION      = os.getenv("FB_API_VERSION", "v18.0")
INSIGHTS_BATCH_SIZE = int(os.getenv("INSIGHTS_BATCH_SIZE", "50"))
MAX_RETRIES         = int(os.getenv("MAX_RETRIES", "5"))
BASE_BACKOFF        = float(os.getenv("BASE_BACKOFF", "1.0"))
HTTP_TIMEOUT_CONNECT  = float(os.getenv("HTTP_TIMEOUT_CONNECT", "30"))
HTTP_TIMEOUT_READ     = float(os.getenv("HTTP_TIMEOUT_READ", "120"))
SLEEP_BETWEEN_CALLS_MIN = float(os.getenv("SLEEP_BETWEEN_CALLS_MIN", "0.05"))
SLEEP_BETWEEN_CALLS_MAX = float(os.getenv("SLEEP_BETWEEN_CALLS_MAX", "0.25"))

CONV_ACTION_KEYS = {
    "link_clicks"               : ["link_click", "inline_link_click", "outbound_click", "outbound_clicks"],
    "website_landing_page_views": ["landing_page_view", "omni_landing_page_view"],
    "add_to_cart"               : ["add_to_cart", "offsite_conversion.fb_pixel_add_to_cart", "omni_add_to_cart"],
    "purchases"                 : ["purchase", "offsite_conversion.fb_pixel_purchase", "omni_purchase"],
    "view_content"              : ["view_content", "offsite_conversion.fb_pixel_view_content", "omni_view_content"],
    "initiate_checkout"         : ["initiate_checkout", "offsite_conversion.fb_pixel_initiate_checkout", "omni_initiated_checkout"],
    "complete_registration"     : ["complete_registration", "offsite_conversion.fb_pixel_complete_registration"],
    "instagram_profile_visits"  : ["instagram_profile_visit"],
    "follows"                   : ["follow", "page_like"],
    "post_reactions"            : ["post_reaction"],
    "comments"                  : ["comment"],
    "post_saves"                : ["onsite_conversion.post_save"],
    "video_views"               : ["video_view"],
}

CONV_API_FIELDS = (
    "ad_id,date_start,"
    "spend,frequency,cpc,cpm,purchase_roas,"
    "actions,"
    "video_p25_watched_actions,video_p50_watched_actions,"
    "video_p75_watched_actions,video_p100_watched_actions,"
    "video_thruplay_watched_actions"
)


# ============================================================
# UTIL
# ============================================================
def chunk_list(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

def daterange(start: date, end_inclusive: date):
    d = start
    while d <= end_inclusive:
        yield d
        d += timedelta(days=1)

def normalize_act_id(s: str) -> str:
    s = (s or "").strip()
    return s if s.startswith("act_") else f"act_{s}"

def is_permission_error(err: dict) -> bool:
    return bool(err) and err.get("code") == 200

def is_rate_limit_error(err: dict) -> bool:
    if not err: return False
    code = err.get("code")
    msg = (err.get("message") or "").lower()
    return code in (4, 17, 32, 613) or "rate limit" in msg or "too many calls" in msg

def is_transient_http_status(status: int) -> bool:
    return status in (408, 429, 500, 502, 503, 504)

def safe_int(v):
    try: return int(float(v))
    except: return 0

def safe_float(v):
    try: return float(v)
    except: return 0.0

def _parse_actions(actions_list):
    out = {}
    for a in (actions_list or []):
        k, v = a.get("action_type"), a.get("value")
        if k and v is not None:
            try: out[k] = float(v)
            except: pass
    return out

def _max_action(actions_map, candidates):
    vals = [actions_map[k] for k in candidates if k in actions_map]
    return int(max(vals)) if vals else 0

def _parse_video_pct_field(field_data):
    for item in (field_data or []):
        if isinstance(item, dict) and item.get("action_type") == "video_view":
            try: return int(float(item.get("value", 0)))
            except: pass
    return 0

def _parse_purchase_roas(v):
    if v is None: return 0.0
    if isinstance(v, list):
        return safe_float(v[0]["value"]) if v else 0.0
    return safe_float(v)


# ============================================================
# DB
# ============================================================
def get_db_conn():
    return pg8000.native.Connection(
        user=os.environ["DB_USER"], password=os.environ["DB_PASSWORD"],
        host=os.environ["DB_HOST"], database=os.environ["DB_NAME"], port=5432
    )

def get_ad_accounts(conn):
    rows = conn.run("SELECT fb_ad_account_id FROM ad_account WHERE fb_ad_account_id IS NOT NULL")
    return [str(r[0]).strip() for r in rows if r and r[0] is not None]

def get_ads_by_account(conn, fb_ad_account_id):
    rows = conn.run("""
        SELECT ad_id, fb_ad_id FROM ad
        WHERE account_id = (SELECT account_id FROM ad_account WHERE fb_ad_account_id = :aid)
        AND fb_ad_id IS NOT NULL
    """, aid=fb_ad_account_id)
    return [{"ad_id": r[0], "fb_ad_id": str(r[1]).strip()} for r in rows]


# ============================================================
# NORMALIZE
# ============================================================
def _normalize_conv(data_rows, forced_date):
    out = []
    for ins in data_rows or []:
        fb_ad_id = str(ins.get("ad_id") or "").strip()
        if not fb_ad_id:
            continue
        am = _parse_actions(ins.get("actions"))
        out.append({
            "fb_ad_id"  : fb_ad_id,
            "date_start": forced_date,  # API date_start 무시, 요청 날짜 강제 사용
            "age"       : ins.get("age") or "unknown",
            "gender"    : ins.get("gender") or "unknown",
            "spend"         : safe_float(ins.get("spend")),
            "frequency"     : safe_float(ins.get("frequency")),
            "cpc"           : safe_float(ins.get("cpc")),
            "cpm"           : safe_float(ins.get("cpm")),
            "purchase_roas" : _parse_purchase_roas(ins.get("purchase_roas")),
            "link_clicks"                : _max_action(am, CONV_ACTION_KEYS["link_clicks"]),
            "website_landing_page_views" : _max_action(am, CONV_ACTION_KEYS["website_landing_page_views"]),
            "add_to_cart"                : _max_action(am, CONV_ACTION_KEYS["add_to_cart"]),
            "purchases"                  : _max_action(am, CONV_ACTION_KEYS["purchases"]),
            "view_content"               : _max_action(am, CONV_ACTION_KEYS["view_content"]),
            "initiate_checkout"          : _max_action(am, CONV_ACTION_KEYS["initiate_checkout"]),
            "complete_registration"      : _max_action(am, CONV_ACTION_KEYS["complete_registration"]),
            "instagram_profile_visits"   : _max_action(am, CONV_ACTION_KEYS["instagram_profile_visits"]),
            "follows"                    : _max_action(am, CONV_ACTION_KEYS["follows"]),
            "post_reactions"             : _max_action(am, CONV_ACTION_KEYS["post_reactions"]),
            "comments"                   : _max_action(am, CONV_ACTION_KEYS["comments"]),
            "post_saves"                 : _max_action(am, CONV_ACTION_KEYS["post_saves"]),
            "video_views"                : _max_action(am, CONV_ACTION_KEYS["video_views"]),
            "video_p25_watched"     : _parse_video_pct_field(ins.get("video_p25_watched_actions")),
            "video_p50_watched"     : _parse_video_pct_field(ins.get("video_p50_watched_actions")),
            "video_p75_watched"     : _parse_video_pct_field(ins.get("video_p75_watched_actions")),
            "video_p100_watched"    : _parse_video_pct_field(ins.get("video_p100_watched_actions")),
            "video_thruplay_watched": _parse_video_pct_field(ins.get("video_thruplay_watched_actions")),
        })
    return out


# ============================================================
# META API
# ============================================================
def fetch_conv_batch(http, access_token, act_id, fb_ad_ids, since, until):
    base_url = f"https://graph.facebook.com/{FB_API_VERSION}/{act_id}/insights"
    params = {
        "access_token"  : access_token,
        "level"         : "ad",
        "breakdowns"    : "age,gender",
        "time_range"    : json.dumps({"since": since, "until": until}),
        "time_increment": "1",
        "fields"        : CONV_API_FIELDS,
        "filtering"     : json.dumps([{"field": "ad.id", "operator": "IN", "value": fb_ad_ids}]),
        "limit"         : "5000",
    }

    def build_url(extra=None):
        p = {**params, **(extra or {})}
        return base_url + "?" + "&".join(f"{k}={quote(str(v))}" for k, v in p.items())

    backoff = BASE_BACKOFF
    for attempt in range(MAX_RETRIES):
        try:
            resp = http.request("GET", build_url())
            data = json.loads(resp.data.decode("utf-8", errors="replace"))
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                s = backoff + random.uniform(0, 0.5)
                print(f"⚠️ Network error (attempt {attempt+1}): {e} → sleep {s:.1f}s")
                time.sleep(s); backoff *= 2; continue
            return [], "FAILED"

        if resp.status == 200 and "data" in data:
            rows = _normalize_conv(data["data"], since)
            while True:
                after = ((data.get("paging") or {}).get("cursors") or {}).get("after")
                if not after: break
                resp2 = http.request("GET", build_url({"after": after}))
                data2 = json.loads(resp2.data.decode("utf-8", errors="replace"))
                if resp2.status != 200 or "data" not in data2: break
                rows.extend(_normalize_conv(data2["data"], since))
                data = data2
            return rows, "OK"

        err = data.get("error") or {}
        if resp.status == 403 and is_permission_error(err):
            return [], "PERMISSION"
        if is_rate_limit_error(err) or is_transient_http_status(resp.status):
            if attempt < MAX_RETRIES - 1:
                s = backoff + random.uniform(0, 0.5)
                time.sleep(s); backoff *= 2; continue
        print(f"⚠️ Non-retryable error status={resp.status}: {str(data)[:200]}")
        return [], "FAILED"
    return [], "FAILED"


# ============================================================
# DB UPSERT
# ============================================================
def upsert_conv(conn, rows, ad_id_map, created_at_utc):
    """전환·참여·영상 지표만 upsert. 기본 성과 지표(reach 등)는 건드리지 않음."""
    saved = skipped = 0
    for rec in rows:
        fb_ad_id = rec.get("fb_ad_id")
        if not fb_ad_id or fb_ad_id not in ad_id_map:
            skipped += 1; continue
        try:
            conn.run("""
                INSERT INTO ad_performance_daily (
                    ad_id, date, age, gender,
                    spend, frequency, cpc, cpm, purchase_roas,
                    link_clicks, website_landing_page_views, add_to_cart, purchases,
                    view_content, initiate_checkout, complete_registration,
                    instagram_profile_visits, follows, post_reactions, comments, post_saves,
                    video_views, video_p25_watched, video_p50_watched, video_p75_watched,
                    video_p100_watched, video_thruplay_watched,
                    created_at, updated_at
                ) VALUES (
                    :ad_id, :date, :age, :gender,
                    :spend, :frequency, :cpc, :cpm, :purchase_roas,
                    :link_clicks, :website_landing_page_views, :add_to_cart, :purchases,
                    :view_content, :initiate_checkout, :complete_registration,
                    :instagram_profile_visits, :follows, :post_reactions, :comments, :post_saves,
                    :video_views, :video_p25_watched, :video_p50_watched, :video_p75_watched,
                    :video_p100_watched, :video_thruplay_watched,
                    :created_at, :updated_at
                )
                ON CONFLICT (ad_id, date, age, gender) DO UPDATE SET
                    spend                        = EXCLUDED.spend,
                    frequency                    = EXCLUDED.frequency,
                    cpc                          = EXCLUDED.cpc,
                    cpm                          = EXCLUDED.cpm,
                    purchase_roas                = EXCLUDED.purchase_roas,
                    link_clicks                  = EXCLUDED.link_clicks,
                    website_landing_page_views   = EXCLUDED.website_landing_page_views,
                    add_to_cart                  = EXCLUDED.add_to_cart,
                    purchases                    = EXCLUDED.purchases,
                    view_content                 = EXCLUDED.view_content,
                    initiate_checkout            = EXCLUDED.initiate_checkout,
                    complete_registration        = EXCLUDED.complete_registration,
                    instagram_profile_visits     = EXCLUDED.instagram_profile_visits,
                    follows                      = EXCLUDED.follows,
                    post_reactions               = EXCLUDED.post_reactions,
                    comments                     = EXCLUDED.comments,
                    post_saves                   = EXCLUDED.post_saves,
                    video_views                  = EXCLUDED.video_views,
                    video_p25_watched            = EXCLUDED.video_p25_watched,
                    video_p50_watched            = EXCLUDED.video_p50_watched,
                    video_p75_watched            = EXCLUDED.video_p75_watched,
                    video_p100_watched           = EXCLUDED.video_p100_watched,
                    video_thruplay_watched       = EXCLUDED.video_thruplay_watched,
                    updated_at                   = :updated_at
            """,
            ad_id=ad_id_map[fb_ad_id], date=rec["date_start"],
            age=rec["age"], gender=rec["gender"],
            spend=rec["spend"], frequency=rec["frequency"],
            cpc=rec["cpc"], cpm=rec["cpm"], purchase_roas=rec["purchase_roas"],
            link_clicks=rec["link_clicks"],
            website_landing_page_views=rec["website_landing_page_views"],
            add_to_cart=rec["add_to_cart"], purchases=rec["purchases"],
            view_content=rec["view_content"], initiate_checkout=rec["initiate_checkout"],
            complete_registration=rec["complete_registration"],
            instagram_profile_visits=rec["instagram_profile_visits"],
            follows=rec["follows"], post_reactions=rec["post_reactions"],
            comments=rec["comments"], post_saves=rec["post_saves"],
            video_views=rec["video_views"],
            video_p25_watched=rec["video_p25_watched"],
            video_p50_watched=rec["video_p50_watched"],
            video_p75_watched=rec["video_p75_watched"],
            video_p100_watched=rec["video_p100_watched"],
            video_thruplay_watched=rec["video_thruplay_watched"],
            created_at=created_at_utc, updated_at=created_at_utc)
            saved += 1
        except Exception as e:
            skipped += 1
            print(f"⚠️ upsert_conv failed fb_ad_id={fb_ad_id} date={rec.get('date_start')}: {e}")
    return saved, skipped


# ============================================================
# MAIN
# ============================================================
def backfill_conversion(start_date: str, end_date: str):
    """전환·참여·영상 지표 backfill. 셀 1과 독립적으로 실행 가능."""
    access_token = os.environ["META_ACCESS_TOKEN"]
    start = datetime.strptime(start_date, "%Y-%m-%d").date()
    end   = datetime.strptime(end_date,   "%Y-%m-%d").date()

    db   = get_db_conn()
    http = urllib3.PoolManager(
        timeout=urllib3.Timeout(connect=HTTP_TIMEOUT_CONNECT, read=HTTP_TIMEOUT_READ),
        retries=False
    )

    try:
        accounts = get_ad_accounts(db)
        print(f"✅ ad_accounts: {len(accounts)}")

        account_ads = []
        for fb_act_id in accounts:
            ads = get_ads_by_account(db, fb_act_id)
            ad_id_map = {a["fb_ad_id"]: a["ad_id"] for a in ads}
            account_ads.append((fb_act_id, normalize_act_id(fb_act_id), list(ad_id_map.keys()), ad_id_map))
        print("✅ Ads loaded.")

        total_saved = total_rows = perm_skipped = 0

        for d in daterange(start, end):
            since = d.strftime("%Y-%m-%d")
            until = (d + timedelta(days=1)).strftime("%Y-%m-%d")
            print(f"\n📅 {since}")
            day_saved = day_rows = 0
            now_utc = datetime.utcnow()

            for fb_act_id, act_id, fb_ad_ids, ad_id_map in account_ads:
                if not fb_ad_ids: continue
                perm_denied = False
                for batch in chunk_list(fb_ad_ids, INSIGHTS_BATCH_SIZE):
                    rows, reason = fetch_conv_batch(http, access_token, act_id, batch, since, until)
                    if reason == "PERMISSION":
                        perm_denied = True; break
                    if rows:
                        saved, _ = upsert_conv(db, rows, ad_id_map, now_utc)
                        day_saved += saved; total_saved += saved
                        day_rows  += len(rows); total_rows += len(rows)
                    time.sleep(random.uniform(SLEEP_BETWEEN_CALLS_MIN, SLEEP_BETWEEN_CALLS_MAX))
                if perm_denied:
                    perm_skipped += 1
                    print(f"  🚫 Permission: {act_id}")

            print(f"  ✅ fetched={day_rows} saved={day_saved}")

        print(f"\n{'='*40}")
        print(f"✅ Conversion backfill done.  total_fetched={total_rows}  total_saved={total_saved}  perm_skipped={perm_skipped}")

    finally:
        try: http.clear()
        except: pass
        try: db.close()
        except: pass


# 실행
backfill_conversion("2025-12-21", "2026-01-01")

✅ ad_accounts: 41
✅ Ads loaded.

📅 2025-12-21
  🚫 Permission: act_399481461969972
  🚫 Permission: act_4241077792781592
  🚫 Permission: act_708046804726689
  🚫 Permission: act_601392272220010
  🚫 Permission: act_911591441881884
  🚫 Permission: act_3747512468894152
  ✅ fetched=3116 saved=3116

📅 2025-12-22
  🚫 Permission: act_399481461969972
  🚫 Permission: act_4241077792781592
  🚫 Permission: act_708046804726689
  🚫 Permission: act_601392272220010
  🚫 Permission: act_911591441881884
  🚫 Permission: act_3747512468894152
  ✅ fetched=3412 saved=3412

📅 2025-12-23
  🚫 Permission: act_399481461969972
  🚫 Permission: act_4241077792781592
  🚫 Permission: act_708046804726689
  🚫 Permission: act_601392272220010
  🚫 Permission: act_911591441881884
  🚫 Permission: act_3747512468894152
  ✅ fetched=3544 saved=3544

📅 2025-12-24
  🚫 Permission: act_399481461969972
  🚫 Permission: act_4241077792781592
  🚫 Permission: act_708046804726689
  🚫 Permission: act_601392272220010
  🚫 Permission: act_911591441

### Campaign / Ad Set / Ad 동기화

In [1]:
import os
import json
import time
import random
import mimetypes
from datetime import datetime, timezone, timedelta
from pathlib import Path
from urllib.parse import quote

import pg8000.native
import urllib3

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

# ============================================================
# CONFIG
# ============================================================
FB_API_VERSION   = os.getenv("FB_API_VERSION", "v21.0")
FB_AD_ACCOUNT_ID = "act_629591912653973"

MAX_RETRIES  = int(os.getenv("MAX_RETRIES", "5"))
BASE_BACKOFF = float(os.getenv("BASE_BACKOFF", "1.0"))

S3_BUCKET   = os.getenv("S3_BUCKET", "").strip()
S3_PREFIX   = os.getenv("S3_PREFIX", "thumbnails").strip().strip("/") or "thumbnails"
S3_BASE_URL = os.getenv("S3_BASE_URL", "").strip()
AWS_REGION  = os.getenv("AWS_REGION", "ap-northeast-2").strip()

# ============================================================
# DB
# ============================================================
def _get_db():
    return pg8000.native.Connection(
        user=os.environ["DB_USER"], password=os.environ["DB_PASSWORD"],
        host=os.environ["DB_HOST"], database=os.environ["DB_NAME"], port=5432
    )

def _get_account_id(conn, fb_ad_account_id: str):
    clean = fb_ad_account_id.replace("act_", "").strip()
    rows = conn.run(
        "SELECT account_id FROM ad_account WHERE fb_ad_account_id = :a OR fb_ad_account_id = :b",
        a=clean, b=f"act_{clean}"
    )
    return rows[0][0] if rows else None

# ============================================================
# META API
# ============================================================
def _api_get(http, access_token, path, params=None):
    url = f"https://graph.facebook.com/{FB_API_VERSION}/{path}"
    p = {"access_token": access_token, "limit": "200", **(params or {})}
    full_url = url + "?" + "&".join(f"{k}={quote(str(v))}" for k, v in p.items())

    backoff = BASE_BACKOFF
    for attempt in range(MAX_RETRIES):
        try:
            resp = http.request("GET", full_url)
            data = json.loads(resp.data.decode("utf-8", errors="replace"))
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                s = backoff + random.uniform(0, 0.5)
                print(f"  ⚠️ 네트워크 오류 (attempt {attempt+1}): {e} → {s:.1f}s")
                time.sleep(s); backoff *= 2; continue
            raise
        if resp.status == 200 and "error" not in data:
            return data
        err = data.get("error") or {}
        code = err.get("code")
        msg  = (err.get("message") or "").lower()
        is_rate = code in (4, 17, 32, 613) or "rate limit" in msg or "too many calls" in msg
        if is_rate or resp.status in (429, 500, 502, 503, 504):
            if attempt < MAX_RETRIES - 1:
                s = backoff + random.uniform(0, 0.5)
                print(f"  ⚠️ Rate limit (attempt {attempt+1}), {s:.1f}s 대기")
                time.sleep(s); backoff *= 2; continue
        raise RuntimeError(f"API 오류 status={resp.status}: {str(data)[:200]}")
    raise RuntimeError(f"{MAX_RETRIES}회 재시도 후 실패")


def _get_all_pages(http, access_token, path, params=None):
    results = []
    data = _api_get(http, access_token, path, params)
    results.extend(data.get("data", []))
    while True:
        after = ((data.get("paging") or {}).get("cursors") or {}).get("after")
        if not after:
            break
        data = _api_get(http, access_token, path, {**(params or {}), "after": after})
        results.extend(data.get("data", []))
    return results

# ============================================================
# S3 THUMBNAIL UPLOAD
# ============================================================
def _s3_public_url(bucket, region, key, base_url=""):
    if base_url:
        return f"{base_url.rstrip('/')}/{key}"
    if region == "us-east-1":
        return f"https://{bucket}.s3.amazonaws.com/{key}"
    return f"https://{bucket}.s3.{region}.amazonaws.com/{key}"


def _guess_ext(content_type: str) -> str:
    ctype = (content_type or "").split(";")[0].strip().lower()
    ext = mimetypes.guess_extension(ctype) if ctype.startswith("image/") else None
    if ext == ".jpe":
        return ".jpg"
    return ext if ext in {".jpg", ".jpeg", ".png", ".webp", ".gif"} else ".jpg"


def _upload_thumbnail(http, s3, fb_ad_id: str, cdn_url: str) -> str | None:
    """CDN URL 이미지를 다운로드해서 S3에 업로드하고 S3 URL을 반환. 실패 시 None."""
    if not cdn_url or not S3_BUCKET:
        return None
    try:
        resp = http.request(
            "GET", cdn_url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=30,
        )
        if resp.status != 200 or not resp.data:
            return None
        content_type = resp.headers.get("Content-Type", "image/jpeg")
        ext = _guess_ext(content_type)
        key = f"{S3_PREFIX}/{fb_ad_id}{ext}"
        s3.put_object(
            Bucket=S3_BUCKET,
            Key=key,
            Body=resp.data,
            ContentType=content_type.split(";")[0].strip() or "image/jpeg",
        )
        return _s3_public_url(S3_BUCKET, AWS_REGION, key, S3_BASE_URL)
    except Exception as e:
        print(f"  ⚠️ S3 업로드 실패 {fb_ad_id}: {e}")
        return None

# ============================================================
# PARSE HELPERS
# ============================================================
_KST = timezone(timedelta(hours=9))

def _parse_ts(ts_str):
    if not ts_str:
        return None
    try:
        dt = datetime.fromisoformat(ts_str.replace("+0000", "+00:00"))
        return dt.astimezone(_KST)
    except Exception:
        return None

def _parse_date(ts_str):
    dt = _parse_ts(ts_str)
    return dt.date() if dt else None

# ============================================================
# UPSERT
# ============================================================
def _upsert_campaigns(conn, campaigns, account_id, now):
    saved = skipped = 0
    for c in campaigns:
        fb_id = str(c.get("id") or "").strip()
        if not fb_id:
            continue
        try:
            conn.run("""
                INSERT INTO campaign
                    (fb_campaign_id, account_id, campaign_name, objective, status, created_time, updated_at)
                VALUES
                    (:fid, :aid, :name, :obj, :status, :created, :now)
                ON CONFLICT (fb_campaign_id) DO UPDATE SET
                    campaign_name = EXCLUDED.campaign_name,
                    objective     = EXCLUDED.objective,
                    status        = EXCLUDED.status,
                    updated_at    = EXCLUDED.updated_at
            """,
            fid=fb_id, aid=account_id, name=c.get("name"),
            obj=c.get("objective"), status=c.get("status"),
            created=_parse_ts(c.get("created_time")), now=now)
            saved += 1
        except Exception as e:
            print(f"  ⚠️ campaign 실패 {fb_id}: {e}")
            skipped += 1
    print(f"  campaign: {saved}건 처리  스킵={skipped}")


def _upsert_ad_sets(conn, ad_sets, campaign_id_map, now):
    saved = skipped = 0
    for s in ad_sets:
        fb_id = str(s.get("id") or "").strip()
        if not fb_id:
            continue
        campaign_id = campaign_id_map.get(str(s.get("campaign_id") or "").strip())
        try:
            conn.run("""
                INSERT INTO ad_set
                    (fb_ad_set_id, campaign_id, ad_set_name, optimization_goal,
                     billing_event, status, created_time, updated_at)
                VALUES
                    (:fid, :cid, :name, :og, :be, :status, :created, :now)
                ON CONFLICT (fb_ad_set_id) DO UPDATE SET
                    campaign_id       = EXCLUDED.campaign_id,
                    ad_set_name       = EXCLUDED.ad_set_name,
                    optimization_goal = EXCLUDED.optimization_goal,
                    billing_event     = EXCLUDED.billing_event,
                    status            = EXCLUDED.status,
                    updated_at        = EXCLUDED.updated_at
            """,
            fid=fb_id, cid=campaign_id, name=s.get("name"),
            og=s.get("optimization_goal"), be=s.get("billing_event"),
            status=s.get("status"), created=_parse_ts(s.get("created_time")), now=now)
            saved += 1
        except Exception as e:
            print(f"  ⚠️ ad_set 실패 {fb_id}: {e}")
            skipped += 1
    print(f"  ad_set: {saved}건 처리  스킵={skipped}")


def _upsert_ads(conn, http, s3, ads, account_id, ad_set_id_map, now):
    saved = skipped = s3_ok = s3_fail = 0
    for a in ads:
        fb_id = str(a.get("id") or "").strip()
        if not fb_id:
            continue
        ad_name      = (a.get("name") or "").strip() or fb_id
        created_date = _parse_date(a.get("created_time"))
        if not created_date:
            skipped += 1; continue

        ad_set_id  = ad_set_id_map.get(str(a.get("adset_id") or "").strip())
        creative   = a.get("creative") or {}
        creative_id  = str(creative.get("id") or "").strip() or None
        body         = creative.get("body")
        cdn_url      = creative.get("thumbnail_url") or creative.get("image_url")
        src_ig       = creative.get("source_instagram_media_id")
        eff_story    = creative.get("effective_object_story_id")
        link_data    = (creative.get("object_story_spec") or {}).get("link_data") or {}
        landing_url  = link_data.get("link")

        # S3에 업로드하고 영구 URL 사용
        thumb_link = _upload_thumbnail(http, s3, fb_id, cdn_url)
        if thumb_link:
            s3_ok += 1
        else:
            s3_fail += 1
            thumb_link = cdn_url  # S3 실패 시 CDN URL 폴백

        try:
            conn.run("""
                INSERT INTO ad (
                    fb_ad_id, account_id, ad_name, status, created_time,
                    creative_id, body, ad_set_id,
                    source_instagram_media_id, effective_object_story_id,
                    thumb_link, landing_page_url, link_url
                ) VALUES (
                    :fid, :aid, :name, :status, :created,
                    :cid, :body, :asid,
                    :src_ig, :eff,
                    :thumb, :lp, :lu
                )
                ON CONFLICT (fb_ad_id) DO UPDATE SET
                    ad_name                   = EXCLUDED.ad_name,
                    status                    = EXCLUDED.status,
                    creative_id               = EXCLUDED.creative_id,
                    body                      = COALESCE(EXCLUDED.body,     ad.body),
                    ad_set_id                 = COALESCE(EXCLUDED.ad_set_id, ad.ad_set_id),
                    source_instagram_media_id = COALESCE(EXCLUDED.source_instagram_media_id, ad.source_instagram_media_id),
                    effective_object_story_id = COALESCE(EXCLUDED.effective_object_story_id, ad.effective_object_story_id),
                    thumb_link                = EXCLUDED.thumb_link,
                    landing_page_url          = COALESCE(EXCLUDED.landing_page_url,  ad.landing_page_url),
                    link_url                  = COALESCE(EXCLUDED.link_url,          ad.link_url)
            """,
            fid=fb_id, aid=account_id, name=ad_name, status=a.get("status"),
            created=created_date, cid=creative_id, body=body, asid=ad_set_id,
            src_ig=src_ig, eff=eff_story,
            thumb=thumb_link, lp=landing_url, lu=landing_url)
            saved += 1
        except Exception as e:
            print(f"  ⚠️ ad 실패 {fb_id}: {e}")
            skipped += 1
    print(f"  ad: {saved}건 처리  스킵={skipped}  S3 성공={s3_ok}  S3 실패={s3_fail}")

# ============================================================
# MAIN
# ============================================================
def sync_campaign_ad(fb_ad_account_id: str):
    act_id = fb_ad_account_id if fb_ad_account_id.startswith("act_") else f"act_{fb_ad_account_id}"
    access_token = os.environ["META_ACCESS_TOKEN"]

    if not S3_BUCKET:
        print("❌ S3_BUCKET 환경변수가 없습니다")
        return

    import boto3
    s3 = boto3.client(
        "s3",
        aws_access_key_id=os.environ.get("AWS_ACCESS_KEY_ID"),
        aws_secret_access_key=os.environ.get("AWS_SECRET_ACCESS_KEY"),
        aws_session_token=os.environ.get("AWS_SESSION_TOKEN"),
        region_name=AWS_REGION,
    )

    db   = _get_db()
    http = urllib3.PoolManager(timeout=urllib3.Timeout(connect=30, read=120), retries=False)

    try:
        account_id = _get_account_id(db, fb_ad_account_id)
        if not account_id:
            print(f"❌ ad_account 테이블에 {fb_ad_account_id} 없음")
            return
        print(f"✅ account_id={account_id}  ({act_id})")

        now = datetime.now(timezone.utc)

        # 1. Campaign
        print("\n📋 Campaign 수집 중...")
        campaigns = _get_all_pages(http, access_token, f"{act_id}/campaigns", {
            "fields": "id,name,objective,status,created_time"
        })
        print(f"  API: {len(campaigns)}건")
        _upsert_campaigns(db, campaigns, account_id, now)

        rows = db.run(
            "SELECT fb_campaign_id, campaign_id FROM campaign WHERE account_id = :aid",
            aid=account_id
        )
        campaign_id_map = {r[0]: r[1] for r in rows}

        # 2. Ad Set
        print("\n📋 Ad Set 수집 중...")
        ad_sets = _get_all_pages(http, access_token, f"{act_id}/adsets", {
            "fields": "id,name,campaign_id,optimization_goal,billing_event,status,created_time"
        })
        print(f"  API: {len(ad_sets)}건")
        _upsert_ad_sets(db, ad_sets, campaign_id_map, now)

        rows = db.run("""
            SELECT s.fb_ad_set_id, s.ad_set_id
            FROM ad_set s
            JOIN campaign c ON s.campaign_id = c.campaign_id
            WHERE c.account_id = :aid
        """, aid=account_id)
        ad_set_id_map = {r[0]: r[1] for r in rows}

        # 3. Ad (thumbnail_width/height로 2048×2048 요청)
        print("\n📋 Ad 수집 중...")
        ads = _get_all_pages(http, access_token, f"{act_id}/ads", {
            "fields": (
                "id,name,status,created_time,adset_id,"
                "creative{id,body,thumbnail_url,image_url,"
                "object_story_spec,effective_object_story_id,"
                "source_instagram_media_id}"
            ),
            "thumbnail_width": "2048",
            "thumbnail_height": "2048",
        })
        print(f"  API: {len(ads)}건  → S3 업로드 중...")
        _upsert_ads(db, http, s3, ads, account_id, ad_set_id_map, now)

        print(f"\n✅ {act_id} 동기화 완료!")

    finally:
        try: http.clear()
        except: pass
        try: db.close()
        except: pass


sync_campaign_ad(FB_AD_ACCOUNT_ID)


✅ account_id=40  (act_629591912653973)

📋 Campaign 수집 중...
  API: 27건
  campaign: 27건 처리  스킵=0

📋 Ad Set 수집 중...
  API: 126건
  ad_set: 126건 처리  스킵=0

📋 Ad 수집 중...
  API: 128건  → S3 업로드 중...
  ad: 128건 처리  스킵=0  S3 성공=128  S3 실패=0

✅ act_629591912653973 동기화 완료!
